**Guess**
1. What fraction of the messages do you think are spam?
    * 1028/5572
2. If a "model" always says ham, what accuracy does it get?
    * 1 − 0.1845 = 0.8155
3. Which will win on F1: TF-IDF + LR, or the embedding network?  By how much?
    * TF-IDF by 0.0698
4. One image/message difference: how many spam messages are in a validation set of
892 messages if 13% are spam, and how many F1 points is ONE missed spam worth?
(Work it out: one message out of about 120 spams.)
   * 892 * 0.13 = 116, 892 - 116 = 776 (non-spam)
   * before: TP = 116, FN = 0 so, precision: 1.0, recall = 1.0, f1-score: 1.0
   * after miss 1 = TP = 116, FN = 1 so, precision: 1.0, recall: 115/116 = 0.9914
   f1-score: 2(1 x 0.9914)/ (1 + 0.9914) = 0.9957
   * Difference: 1 - 0.9957 = 0.0043


In [2]:
import re, copy, numpy as np, pandas as pd, torch, torch.nn as nn
from collections import Counter
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix


In [3]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", header=None, names=["label", "text"])
df["y"] = (df["label"] == "spam").astype(int)
print(df.shape)
print(df["y"].value_counts(), df["y"].mean())
print(df.sample(5, random_state=1))


(5572, 3)
y
0    4825
1     747
Name: count, dtype: int64 0.13406317300789664
     label                                               text  y
1078   ham                       Yep, by the pretty sculpture  0
4028   ham      Yes, princess. Are you going to make me moan?  0
958    ham                         Welp apparently he retired  0
4642   ham                                            Havent.  0
4674   ham  I forgot 2 ask ü all smth.. There's a card on ...  0


In [4]:
base_acc = 1 - df["y"].mean()
print("baseline accuracy:", round(base_acc, 4))

baseline accuracy: 0.8659


In [7]:
X_tr_all, X_te, y_tr_all, y_te = train_test_split(
    df["text"], df["y"], test_size=0.2, stratify=df["y"], random_state=42
)

X_tr, X_va, y_tr, y_va = train_test_split(
    X_tr_all, y_tr_all, test_size=0.2, stratify=y_tr_all, random_state=42
)

print(len(X_tr_all),  len(X_te), len(X_tr), len(X_va), int(y_va.sum()))

4457 1115 3565 892 120


In [13]:
def tokenise(t):
    return re.findall(r"[a-z0-9']+", t.lower())

In [14]:
cnt = Counter(w for t in X_tr for w in tokenise(t))
itos = ["<PAD>", "<UNK>"] + sorted(w for w, c in  cnt.items() if c >= 2)
stoi = {w: i for i, w in enumerate(itos)}
print("vocab size", len(itos), "of", len(cnt), "distinct training words")

vocab size 3332 of 6981 distinct training words


In [18]:
lens = X_tr.map(lambda t: len(tokenise(t)))
print(lens.mean(), np.percentile(lens, 95), lens.max())

15.733239831697055 33.0 121


In [ ]:
L = 40
def encode(t):
    ids = [stoi.get(w, 1) for w in tokenise(t)[:L]]
    return ids + [0] * (L - len(ids))

def to_tensors(X, y):
    return (torch.tensor([encode(t) for t in X]),
            torch.tensor(y.values, dtype=torch.float32).unsqueeze(1))

In [17]:
Xt, yt = to_tensors(X_tr, y_tr)
Xv, yv = to_tensors(X_va, y_va)
print(Xt.shape, yt.shape, Xt[0][:8])

torch.Size([3565, 40]) torch.Size([3565, 1]) tensor([2855, 2755, 2647, 3184,  977, 3314, 1338, 2219])


In [19]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cands = {
    "count + LR": Pipeline([("v", CountVectorizer()), ("m", LogisticRegression(max_iter=1000))]),
    "tfidf + LR": Pipeline([("v", TfidfVectorizer()), ("m", LogisticRegression(max_iter=1000))]),
    "tfidf + LR, C=10": Pipeline([("v", TfidfVectorizer()), ("m", LogisticRegression(C=10, max_iter=1000))]),
    "tfidf 1-2gram + LR, C=10": Pipeline([("v", TfidfVectorizer(ngram_range=(1, 2))), ("m", LogisticRegression(C=10, max_iter=1000))]),
    "tfidf + LR balanced, C=10": Pipeline([("v", TfidfVectorizer()), ("m", LogisticRegression(C=10, class_weight="balanced", max_iter=1000))])
}

In [24]:
rows = []
for name, pipe in cands.items():
    r = cross_validate(pipe, X_tr_all, y_tr_all, cv=skf, scoring="f1", return_train_score=True)
    m, s, t = r["test_score"].mean(), r["test_score"].std(), r["train_score"].mean()
    rows.append((name, round(m, 4), round(s, 4), round(t, 4), round(t - m, 4)))
cv_table = pd.DataFrame(rows, columns=["model", "cv_f1", "std", "train_f1", "gap"])
cv_table

,model,cv_f1,std,train_f1,gap
0,count + LR,0.9218,0.0205,0.9924,0.0706
1,tfidf + LR,0.8528,0.0199,0.8827,0.0299
2,"tfidf + LR, C=10",0.9332,0.0066,0.9939,0.0607
3,"tfidf 1-2gram + LR, C=10",0.9434,0.0079,0.9998,0.0564
4,"tfidf + LR balanced, C=10",0.9444,0.0105,0.9960,0.0516


In [27]:
val_rows = []
for name, pipe in cands.items():
    pipe.fit(X_tr, y_tr)
    p = pipe.predict(X_va)
    val_rows.append((name, round(f1_score(y_va, p), 4), round(precision_score(y_va, p), 4), round(recall_score(y_va, p), 4)))
pd.DataFrame(val_rows, columns=["model", "val_f1", "precision", "recall"])


,model,val_f1,precision,recall
0,count + LR,0.9018,0.9712,0.8417
1,tfidf + LR,0.8476,0.9889,0.7417
2,"tfidf + LR, C=10",0.9156,0.9810,0.8583
3,"tfidf 1-2gram + LR, C=10",0.9351,0.9730,0.9000
4,"tfidf + LR balanced, C=10",0.9185,0.9469,0.8917


In [29]:
class AvgEmb(nn.Module):
    def __init__(self, V, d=32):
        super().__init__()
        self.emb = nn.Embedding(V, d, padding_idx=0)
        self.out = nn.Linear(d, 1)

    def forward(self, x):
        mask = (x != 0).unsqueeze(-1).float()
        e = self.emb(x) * mask
        avg = e.sum(1) / mask.sum(1).clamp(min=1)
        return self.out(avg)

print(sum(p.numel() for p in AvgEmb(len(itos)).parameters()))

106657


In [31]:
def run(seed=42, epochs=60, patience=6):
    torch.manual_seed(seed)
    model = AvgEmb(len(itos))
    opt = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)
    lf = nn.BCEWithLogitsLoss()
    g = torch.Generator().manual_seed(seed)
    best_vl, best_state, best_ep, bad = 1e9, None, 0, 0
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(Xt), 64):
            idx = perm[i:i + 64]
            opt.zero_grad()
            loss = lf(model(Xt[idx]), yt[idx])
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = lf(model(Xv), yv).item()
        if vl < best_vl - 1e-4:
            best_vl, best_state, best_ep, bad = vl, copy.deepcopy(model.state_dict()), ep, 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        p = (torch.sigmoid(model(Xv)) > 0.5).int().squeeze(1).numpy()
    return model, best_ep, best_vl, f1_score(y_va, p), precision_score(y_va, p), recall_score(y_va, p)

In [32]:
nn_rows = []
for sd in [42, 1, 2]:
    _, be, bl, f1, pr, rc = run(sd)
    nn_rows.append((sd, be, round(bl, 4), round(f1, 4), round(pr, 4), round(rc, 4)))
pd.DataFrame(nn_rows, columns=["seed", "best_epoch", "val_loss", "val_f1", "precision", "recall"])

,seed,best_epoch,val_loss,val_f1,precision,recall
0,42,5,0.0725,0.9198,0.9316,0.9083
1,1,5,0.0748,0.9060,0.9298,0.8833
2,2,4,0.0750,0.9185,0.9469,0.8917


**Choose before the test set**

**Decision (written before opening the test set)**

**Rule:** two models are different only if |difference of CV means| > max(std1, std2); otherwise they tie.
**Noise:** one missed spam in validation is worth about 0.0043 F1, so differences under about 3 messages (~0.013) are noise.

| model | CV F1 ± std | train−CV gap | val F1 | val precision | val recall |
|---|---|---|---|---|---|
| count + LR | 0.9218 ± 0.0205 | 0.0706 | 0.9018 | 0.9712 | 0.8417 |
| tfidf + LR | 0.8528 ± 0.0199 | 0.0299 | 0.8476 | 0.9889 | 0.7417 |
| tfidf + LR, C=10 | 0.9332 ± 0.0066 | 0.0607 | 0.9156 | 0.9810 | 0.8583 |
| tfidf 1-2gram + LR, C=10 | 0.9434 ± 0.0079 | 0.0564 | 0.9351 | 0.9730 | 0.9000 |
| tfidf + LR balanced, C=10 | 0.9444 ± 0.0105 | 0.0516 | 0.9185 | 0.9469 | 0.8917 |
| PyTorch AvgEmb (3 seeds) | not cross-validated | n/a | 0.906 to 0.920 | 0.930 to 0.947 | 0.883 to 0.908 |
-----------------------------------------------------------------------------------------------------------------------
| Pair | difference of CV means | max(std1, std2) | Result |
|---|---|---|---|
| balanced vs 1-2gram | 0.0010 | 0.0105 | tie |
| 1-2gram vs tfidf + LR, C=10 | 0.0102 | 0.0079 | different |
| balanced vs tfidf + LR, C=10 | 0.0112 | 0.0105 | different (just) |
| balanced (best) vs count + LR | 0.0226 | 0.0205 | different (just) |

**Decision: tfidf + LR balanced, C=10.**
- It has the highest CV mean (0.9444) and ties with the 1-2gram model (gap 0.0010, far below the 0.0105 fold-to-fold noise). Between the two tied models I take the one with fewer columns: it keeps the unigram vocabulary and only adds a class weight.
- The cost of this choice: its std is larger (0.0105 vs 0.0079), and on validation the 1-2gram model is ahead by 0.0166 (3 to 4 messages), which is at the edge of the noise level and does not overturn the CV tie.
- The network's validation F1 (0.906 to 0.920) contains the balanced model's 0.9185, so it is a tie. The sklearn model is deterministic, faster and needs no seeds, so it is preferred.
- If a missed spam were much more expensive than a false alarm, I would lower the decision threshold below 0.5 (chosen on the validation set); class_weight="balanced" already pushes in the same direction.

In [34]:
best = cands["tfidf + LR balanced, C=10"]
best.fit(X_tr_all, y_tr_all)
pt = best.predict(X_te)
print("F1", round(f1_score(y_te, pt), 4), "precision", round(precision_score(y_te, pt), 4), "recall", round(recall_score(y_te, pt), 4))
print(confusion_matrix(y_te, pt))

F1 0.9485 precision 0.9718 recall 0.9262
[[962   4]
 [ 11 138]]


In [36]:
vec, lr = best.named_steps["v"], best.named_steps["m"]
names = np.array(vec.get_feature_names_out())
order = np.argsort(lr.coef_[0])
print("most spammy:", list(names[order[-10:]][::-1]))
print("most hammy:", list(names[order[:10]]))

wrong = X_te[pt != y_te]
for text, yy, pp in zip(wrong, y_te[pt != y_te], pt[pt != y_te]):
    print(yy, pp, text[:90])

most spammy: ['txt', 'uk', 'text', 'reply', 'stop', 'call', '150p', 'www', 'claim', 'service']
most hammy: ['me', 'my', 'gt', 'lt', 'can', 'ok', 'll', 'that', 'sir', 'he']
0 1 Alright i have a new goal now
1 0 Dear Voucher Holder 2 claim your 1st class airport lounge passes when using Your holiday v
1 0 ringtoneking 84484
1 0 Sorry I missed your call let's talk when you have the time. I'm on 07090201529
1 0 Latest News! Police station toilet stolen, cops have nothing to go on!
0 1 K..u also dont msg or reply to his msg..
1 0 Can U get 2 phone NOW? I wanna chat 2 set up meet Call me NOW on 09096102316 U can cum her
1 0 For sale - arsenal dartboard. Good condition but no doubles or trebles!
0 1 Ok... Ur typical reply...
1 0 Email AlertFrom: Jeri StewartSize: 2KBSubject: Low-cost prescripiton drvgsTo listen to ema
1 0 Download as many ringtones as u like no restrictions, 1000s 2 choose. U can even send 2 yr
1 0 ROMCAPspam Everyone around should be responding well to your presence since yo

**Paper Exercise**
H1  Shape and parameter trace of AvgEmb.forward for a batch of 64 messages, with a
    vocabulary of 3,332 words and d = 32.  Write the shape after each line:
   * x -> (64, 40)  
   * emb(x) ->  (64, 40, 32)
   * mask ->  (64, 40, 1)
   * e ->  (64, 40, 32)
   * e.sum(1) ->  (64, 32)
   * mask.sum(1) ->  (64, 1)
   * avg ->  (64, 32)
   * out(avg) -> (64, 1)
    Then the parameter count: Embedding + Linear.  (If your tokenise is the same as
    above, the printed number should agree with yours.)
   * Embedding: 3332 x 32 = 106,624 parameters
   * Linear: 32 x 1 + 1(bias) = 33
   * Total parameters: 106,657

H2  A confusion matrix on a test set of 1,115 messages (this is the layout
    [[TN, FP], [FN, TP]]):    [[963, 3], [16, 133]]

       a) accuracy: (963 + 133) / 1115 = 1096 / 1115 = 0.9830
       b) precision of spam: TP / (TP + FP) = 133 / 136 = 0.9779
       c) recall of spam: TP / (TP + FN) = 133 / 149 = 0.8926
       d) F1: 2PR / (P + R) = 2 x 0.9779 x 0.8926 / (0.9779 + 0.8926) = 0.9333
       e) the baseline accuracy of "always ham" on this test set (ham = TN + FP),
          and one sentence: why is 0.983 not as impressive as it looks?
             * ham = 963 + 3 = 966 of 1115 -> 0.8664.  0.983 is only 11.7 points above doing
               nothing, and it hides that 16 of 149 spams (10.7%) got through.
       f) which error is more expensive for a bank (fraud), FP or FN?  What would you
          change to trade one for the other?  (Hint: the 0.5 in `> 0.5`.)
             * For a bank, FN (a missed fraud) costs more.  Lower the threshold (for example
               predict spam when p > 0.3 instead of 0.5): recall goes up, precision goes down.
               Choose the threshold on the validation set, never on the test set.

H3  Why does TfidfVectorizer(ngram_range=(1, 2)) help with "not good" and "call now"?
    What does it cost (think of the number of columns)?
    
    * A bigram is a pair of neighbouring words ("not good", "call now") as ONE feature, so
    a little word order and negation come back.  Cost: many more columns (every
    distinct pair), more overfitting risk (train F1 0.9998), a bigger matrix.


**QUESTIONS FOR YOUR NOTES  (S19-1 to S19-8)**
- S19-1  Why is accuracy a bad metric here?  Which metric do you report instead?
    * Because 87% is ham messages and it is already scores 0.87 "always ham" without learning nothing. 
    I would report to precision, recall and f1-score with confusion matrix.
- S19-2  Why must the TF-IDF vectoriser be inside the Pipeline?
    * IDF is computed from the data.  Inside the Pipeline it is learned on each training
    fold only; outside, it would use the validation messages (leakage), and the CV
    score would be too optimistic.
- S19-3  Why is the vocabulary built from the training text only?  What does <UNK> do?
    * To prevent from data leakage. <UNK> (id 1) stands for any
    word that is not in the vocabulary, so the model still gets an input.
- S19-4  "tfidf + LR" with the default C got a much lower F1 than with C=10.  In your
       own words: what does a small C do (Day 8 regularisation), and why is that
       harmful when the TF-IDF numbers are small?
    *  small C like default applies heavy regularization. It heavily penalizes the model for having large 
    coefficient weights, forcing the weights to shrink aggressively toward zero to prevent overfitting.
    It is harmful when the tfidf numbers are small because the model will underfits heavily, 
    predicting almost everything as "ham," which destroys your Recall for spam and tanks your overall F1 score.
- S19-5  What do padding_idx=0 and the mask do in AvgEmb?
    * padding_idx=0 keeps the PAD row of the table at zero and does not train it.  The mask
    zeros the PAD vectors and divides by the number of REAL tokens, so the average ignores the filler.
- S19-6  Why does the PyTorch model reach its best validation loss after only a few
       epochs?  (Think: 3,332 x 32 = 106,624 weights, and 3,565 training messages.)
    * The network has about 107k weights for 3.5k messages: it can memorise the training
    messages after a few epochs, validation loss then rises, and early stopping saves the
    best epoch.
- S19-7  Look at the messages the model missed (Cell 11).  What do the missed spams have
       in common?
    * Many look like normal text or very short; some use words the vocabulary has barely seen
    (ringtone, prescripiton typo); the model only counts words and ignores context.
- S19-8  What do you change if the business says "missing a spam is much worse than a
       false alarm"?  Name the knob.
    * I would reduce the threshold or use class_weight="balanced" or a higher weight for spam class
    so i can detect more spam and false alarm. Pick it on the validation set.
